# Complete Guide to Backtesting Libraries in Python

---

## What is Backtesting?

Backtesting = **"What if I had run this strategy in the past?"**

You feed historical data into your strategy, simulate trades, and measure how much money you would have made (or lost).

```
Historical Data → Your Strategy Logic → Simulated Trades → Performance Report
```

### Why Can't We Just Use a For Loop?

You *could* write backtesting from scratch with pandas, but dedicated libraries handle:
- **Order execution simulation** (market orders, limit orders, slippage)
- **Commission/fees** (realistic transaction costs)
- **Position tracking** (how many shares you own, cash balance)
- **Performance metrics** (Sharpe, drawdown, etc.)
- **Visualization** (equity curves, trade logs)

### Libraries We'll Cover

| Library | Style | Best For | Difficulty |
|---------|-------|----------|------------|
| **Manual (pandas)** | DIY | Understanding the basics | Easiest |
| **Backtrader** | Event-driven | Complex strategies, multiple data feeds | Medium |
| **vectorbt** | Vectorized | Fast iteration, parameter optimization | Medium |
| **backtesting.py** | Lightweight | Quick prototyping, simple strategies | Easiest |
| **Zipline** | Event-driven | Production-grade, Quantopian-style | Hardest |

### Two Approaches to Backtesting

**1. Vectorized backtesting** (vectorbt, pandas)
- Process ALL data at once using array operations
- Very fast (seconds for millions of rows)
- Limited: hard to model complex order logic

**2. Event-driven backtesting** (Backtrader, Zipline)
- Process data **bar by bar** (one day at a time)
- Slower but more realistic
- Can model complex logic: stop-losses, trailing stops, portfolio rebalancing

In [ ]:
# Install all backtesting libraries
!pip install -q yfinance pandas numpy matplotlib
!pip install -q backtrader
!pip install -q vectorbt
!pip install -q backtesting

print('All libraries installed!')

In [ ]:
# Download sample data that we'll use throughout this notebook

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

# Download Apple and Microsoft for single-stock examples
# Download 5 stocks for portfolio examples
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'XOM']

data = yf.download(TICKERS, start='2020-01-01', end='2025-12-31', auto_adjust=False)

# Single stock for simple examples
aapl = yf.download('AAPL', start='2020-01-01', end='2025-12-31', auto_adjust=False)

print(f'Data downloaded: {data.shape}')
print(f'Date range: {data.index[0].date()} to {data.index[-1].date()}')
print(f'AAPL data: {aapl.shape}')
aapl.head(3)

---
---
# Part 1: Manual Backtesting with Pandas
---
---

Before using any library, let's build a backtest **from scratch** so you understand what's happening under the hood.

## Strategy: Simple Moving Average (SMA) Crossover

This is the "Hello World" of trading strategies:

- Calculate the **50-day SMA** (short-term trend) and **200-day SMA** (long-term trend)
- **BUY** when the 50-day crosses ABOVE the 200-day ("Golden Cross" — short-term momentum is rising)
- **SELL** when the 50-day crosses BELOW the 200-day ("Death Cross" — momentum is falling)

```
50-day SMA goes ABOVE 200-day SMA → BUY signal
50-day SMA goes BELOW 200-day SMA → SELL signal
```

### What is SMA?
SMA = average closing price over the last N days.

Example: 5-day SMA = (price today + yesterday + ... + 4 days ago) / 5

It "smooths out" the daily noise and shows the trend.

In [ ]:
# ===================================================================
# MANUAL BACKTEST — Step by Step
# ===================================================================

# Step 1: Calculate the signals
df = aapl[['Close']].copy()
df.columns = ['close']  # Flatten column names

# Moving averages
df['sma_50'] = df['close'].rolling(50).mean()
df['sma_200'] = df['close'].rolling(200).mean()

# Signal: 1 when 50-day > 200-day (bullish), 0 when not
df['signal'] = (df['sma_50'] > df['sma_200']).astype(int)

# Position: shift signal by 1 day (you can only act on TOMORROW, not today)
# This prevents look-ahead bias — you see today's signal, trade tomorrow
df['position'] = df['signal'].shift(1)

# Daily return of the stock
df['stock_return'] = df['close'].pct_change()

# Strategy return = stock return * position (1 if holding, 0 if not)
df['strategy_return'] = df['stock_return'] * df['position']

df = df.dropna()

print('Signal DataFrame:')
print('  signal=1 means we HOLD the stock (50-day SMA > 200-day SMA)')
print('  signal=0 means we are OUT (in cash)')
df[['close', 'sma_50', 'sma_200', 'signal', 'position']].tail(5)

In [ ]:
# Step 2: Calculate cumulative returns

df['cum_stock'] = (1 + df['stock_return']).cumprod()       # Buy & Hold
df['cum_strategy'] = (1 + df['strategy_return']).cumprod() # Our strategy

# Step 3: Plot
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

# Top: Cumulative returns
axes[0].plot(df.index, df['cum_stock'], label='Buy & Hold AAPL', linewidth=2)
axes[0].plot(df.index, df['cum_strategy'], label='SMA Crossover Strategy', linewidth=2)
axes[0].set_title('Manual Backtest: SMA 50/200 Crossover on AAPL', fontsize=16)
axes[0].set_ylabel('Growth of $1')
axes[0].legend(fontsize=12)

# Bottom: Position (in/out)
axes[1].fill_between(df.index, df['position'], alpha=0.3, color='green', label='In Market')
axes[1].set_ylabel('Position')
axes[1].set_xlabel('Date')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Out (Cash)', 'In (Holding)'])
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Performance comparison
total_stock = (df['cum_stock'].iloc[-1] - 1) * 100
total_strat = (df['cum_strategy'].iloc[-1] - 1) * 100
print(f'Buy & Hold return: {total_stock:.1f}%')
print(f'Strategy return:   {total_strat:.1f}%')
print(f'\nGreen shading = periods when the strategy was in the market.')
print(f'White gaps = periods when it was sitting in cash (avoiding drawdowns).')

In [ ]:
# Step 4: Calculate performance metrics manually
# This is what backtesting libraries do automatically

def manual_metrics(returns, name='Strategy'):
    """Calculate key performance metrics from daily returns."""
    total = (1 + returns).prod() - 1
    n_years = len(returns) / 252
    annual_ret = (1 + total) ** (1 / n_years) - 1
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = (annual_ret - 0.04) / annual_vol  # 4% risk-free rate
    
    cum = (1 + returns).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    
    win_rate = (returns[returns != 0] > 0).mean()  # Exclude days with 0 return (out of market)
    
    print(f'--- {name} ---')
    print(f'Total Return:     {total*100:.1f}%')
    print(f'Annual Return:    {annual_ret*100:.1f}%')
    print(f'Annual Volatility:{annual_vol*100:.1f}%')
    print(f'Sharpe Ratio:     {sharpe:.3f}')
    print(f'Max Drawdown:     {max_dd*100:.1f}%')
    print(f'Win Rate:         {win_rate*100:.1f}%')
    print()

manual_metrics(df['stock_return'], 'Buy & Hold AAPL')
manual_metrics(df['strategy_return'], 'SMA Crossover')

### Limitations of Manual Backtesting

Our manual backtest above is missing:
- **Transaction costs** (commissions, slippage)
- **Partial fills** (what if you can't buy all shares at once?)
- **Cash management** (tracking exact dollar amounts)
- **Multiple assets** (portfolio management)
- **Order types** (limit orders, stop-loss orders)

This is why we use backtesting libraries. Let's now rebuild the same strategy in each library.

---
---
# Part 2: Backtrader
---
---

## Overview

**Backtrader** is the most popular Python backtesting framework. It's **event-driven** — it processes data one bar (day) at a time, just like real trading.

### Key Concepts

| Concept | What It Is | Analogy |
|---------|-----------|----------|
| **Cerebro** | The "brain" — orchestrates everything | The game engine |
| **Strategy** | Your trading logic | The player's AI |
| **Data Feed** | Historical price data | The game world |
| **Broker** | Simulates order execution, tracks cash | The bank |
| **Order** | A buy/sell instruction | A move in the game |
| **Indicator** | Technical calculations (SMA, RSI, etc.) | Player's sensors |
| **Analyzer** | Calculates performance metrics | The scoreboard |

### How Backtrader Works

```
1. Create Cerebro (brain)
2. Add Data Feed (historical prices)
3. Add Strategy (your trading rules)
4. Set Broker parameters (cash, commission)
5. Run → Cerebro feeds data to Strategy one day at a time
6. Strategy decides to buy/sell → Broker executes
7. Analyze results
```

### The Strategy Class

Every Backtrader strategy has these key methods:

| Method | When It Runs | What You Do |
|--------|-------------|-------------|
| `__init__` | Once at start | Set up indicators |
| `next` | Every bar (every day) | Check signals, place orders |
| `notify_order` | When an order status changes | Log trades (optional) |
| `notify_trade` | When a trade completes | Log P&L (optional) |

In [ ]:
# ===================================================================
# BACKTRADER — Example 1: SMA Crossover (Single Stock)
# ===================================================================

import backtrader as bt

# Step 1: Define the Strategy
class SMACrossover(bt.Strategy):
    """
    Buy when 50-day SMA crosses above 200-day SMA.
    Sell when 50-day SMA crosses below 200-day SMA.
    """
    
    # Parameters — these can be tuned later
    params = (
        ('fast_period', 50),   # Short-term moving average
        ('slow_period', 200),  # Long-term moving average
    )
    
    def __init__(self):
        # __init__ runs ONCE at the start
        # Set up indicators here (Backtrader calculates them automatically)
        self.sma_fast = bt.indicators.SMA(self.data.close, period=self.params.fast_period)
        self.sma_slow = bt.indicators.SMA(self.data.close, period=self.params.slow_period)
        
        # CrossOver indicator: +1 when fast crosses above slow, -1 when below
        self.crossover = bt.indicators.CrossOver(self.sma_fast, self.sma_slow)
        
        # Track trade count
        self.trade_count = 0
    
    def next(self):
        # next() runs EVERY DAY
        # self.data.close[0] = today's close price
        # self.position = current position (True if we own shares)
        
        if self.crossover > 0:  # Fast SMA crossed ABOVE slow SMA
            if not self.position:  # Only buy if we don't already own
                self.buy()         # Buy with all available cash
                self.trade_count += 1
        
        elif self.crossover < 0:  # Fast SMA crossed BELOW slow SMA
            if self.position:      # Only sell if we own shares
                self.sell()        # Sell all shares
                self.trade_count += 1
    
    def stop(self):
        # stop() runs ONCE at the end
        print(f'Total trades: {self.trade_count}')

print('Strategy class defined.')
print('Key methods: __init__ (setup), next (daily logic), stop (final summary)')

In [ ]:
# Step 2: Set up and run the backtest

# Create the brain
cerebro = bt.Cerebro()

# Add data (AAPL)
bt_data = bt.feeds.PandasData(dataname=aapl)
cerebro.adddata(bt_data)

# Add strategy
cerebro.addstrategy(SMACrossover)

# Set broker parameters
cerebro.broker.setcash(100_000)           # Starting cash
cerebro.broker.setcommission(commission=0.001)  # 0.1% per trade

# Set position sizing — use 95% of cash per trade (leave 5% buffer)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)

# Add analyzers (automatic performance measurement)
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.04/252)
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')

# Run!
print(f'Starting Portfolio Value: ${cerebro.broker.getvalue():,.2f}')
results = cerebro.run()
print(f'Final Portfolio Value:    ${cerebro.broker.getvalue():,.2f}')
print(f'Return:                  {(cerebro.broker.getvalue()/100000 - 1)*100:.1f}%')

In [ ]:
# Step 3: Extract analyzer results

strat = results[0]

# Sharpe Ratio
sharpe = strat.analyzers.sharpe.get_analysis()
print(f'Sharpe Ratio: {sharpe.get("sharperatio", "N/A")}')

# Drawdown
dd = strat.analyzers.drawdown.get_analysis()
print(f'Max Drawdown: {dd.max.drawdown:.2f}%')
print(f'Max Drawdown Duration: {dd.max.len} days')

# Trade statistics
trades = strat.analyzers.trades.get_analysis()
total_trades = trades.total.total if hasattr(trades.total, 'total') else 0
print(f'\nTotal Trades: {total_trades}')
if hasattr(trades, 'won') and hasattr(trades.won, 'total'):
    print(f'Winning Trades: {trades.won.total}')
    print(f'Losing Trades: {trades.lost.total}')
    print(f'Win Rate: {trades.won.total / total_trades * 100:.1f}%')

In [ ]:
# Step 4: Plot the results (Backtrader has built-in plotting)
# This shows: price, indicators, buy/sell markers, portfolio value

cerebro.plot(style='candlestick', volume=False,
             figsize=(16, 10))
plt.show()

print('Green arrows = BUY, Red arrows = SELL')
print('Top panel = price + indicators, Bottom panel = portfolio value')

## Backtrader — Example 2: RSI Mean Reversion Strategy

**Strategy logic:**
- Buy when RSI drops below 30 (oversold — stock fell too much, likely to bounce)
- Sell when RSI rises above 70 (overbought — stock rose too much, likely to drop)

This is a **mean reversion** strategy — it bets that extreme moves will reverse.

In [ ]:
class RSIMeanReversion(bt.Strategy):
    params = (
        ('rsi_period', 14),
        ('rsi_low', 30),    # Buy when RSI drops below this
        ('rsi_high', 70),   # Sell when RSI rises above this
    )
    
    def __init__(self):
        self.rsi = bt.indicators.RSI(self.data.close, period=self.params.rsi_period)
    
    def next(self):
        if self.rsi < self.params.rsi_low:  # Oversold
            if not self.position:
                self.buy()
        
        elif self.rsi > self.params.rsi_high:  # Overbought
            if self.position:
                self.sell()

# Run backtest
cerebro2 = bt.Cerebro()
cerebro2.adddata(bt.feeds.PandasData(dataname=aapl))
cerebro2.addstrategy(RSIMeanReversion)
cerebro2.broker.setcash(100_000)
cerebro2.broker.setcommission(commission=0.001)
cerebro2.addsizer(bt.sizers.PercentSizer, percents=95)
cerebro2.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.04/252)
cerebro2.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

print(f'Start: ${cerebro2.broker.getvalue():,.2f}')
results2 = cerebro2.run()
print(f'End:   ${cerebro2.broker.getvalue():,.2f}')
print(f'Return: {(cerebro2.broker.getvalue()/100000-1)*100:.1f}%')

s2 = results2[0]
print(f'Sharpe: {s2.analyzers.sharpe.get_analysis().get("sharperatio", "N/A")}')
print(f'Max DD: {s2.analyzers.drawdown.get_analysis().max.drawdown:.1f}%')

## Backtrader — Example 3: Multi-Stock Portfolio Strategy

This is what you need for the **group project** — managing multiple stocks at once.

**Strategy:** Every month, check RSI for all stocks. Buy the most oversold ones, sell the overbought ones.

In [ ]:
class MultiStockRSI(bt.Strategy):
    """
    Monthly rebalancing portfolio strategy.
    Allocates more weight to oversold stocks (low RSI).
    """
    params = (
        ('rebalance_days', 21),  # Rebalance every ~21 trading days (monthly)
        ('rsi_period', 14),
    )
    
    def __init__(self):
        # Create RSI indicator for EACH stock
        self.rsi = {}
        for d in self.datas:
            self.rsi[d._name] = bt.indicators.RSI(d.close, period=self.params.rsi_period)
        
        self.counter = 0  # Day counter for rebalancing
    
    def next(self):
        self.counter += 1
        
        # Only rebalance every N days
        if self.counter % self.params.rebalance_days != 0:
            return
        
        # Score each stock by RSI (lower RSI = higher score = more attractive)
        scores = {}
        for d in self.datas:
            rsi_val = self.rsi[d._name][0]  # Current RSI value
            # Invert RSI: low RSI (oversold) → high score
            scores[d._name] = 100 - rsi_val
        
        # Normalize scores to get weights
        total_score = sum(scores.values())
        if total_score == 0:
            return
        
        weights = {k: v / total_score for k, v in scores.items()}
        
        # Rebalance: adjust each position to target weight
        for d in self.datas:
            target = weights.get(d._name, 0)
            self.order_target_percent(d, target=target)

# Set up multi-stock backtest
cerebro3 = bt.Cerebro()

# Add each stock as a separate data feed
for ticker in TICKERS:
    ticker_data = data.xs(ticker, axis=1, level=1).dropna() if isinstance(data.columns, pd.MultiIndex) else data
    # Download individually to avoid MultiIndex issues
    ticker_df = yf.download(ticker, start='2020-01-01', end='2025-12-31', auto_adjust=False)
    feed = bt.feeds.PandasData(dataname=ticker_df, name=ticker)
    cerebro3.adddata(feed)

cerebro3.addstrategy(MultiStockRSI)
cerebro3.broker.setcash(100_000)
cerebro3.broker.setcommission(commission=0.001)
cerebro3.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.04/252)
cerebro3.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
cerebro3.addanalyzer(bt.analyzers.Returns, _name='returns')

print(f'Start: ${cerebro3.broker.getvalue():,.2f}')
results3 = cerebro3.run()
print(f'End:   ${cerebro3.broker.getvalue():,.2f}')
print(f'Return: {(cerebro3.broker.getvalue()/100000-1)*100:.1f}%')

s3 = results3[0]
print(f'Sharpe: {s3.analyzers.sharpe.get_analysis().get("sharperatio", "N/A")}')
print(f'Max DD: {s3.analyzers.drawdown.get_analysis().max.drawdown:.1f}%')

## Backtrader — Example 4: Strategy with Stop-Loss and Take-Profit

Real strategies use **risk management orders**:

- **Stop-Loss**: "If the price drops 5% from my buy price, sell automatically to limit losses"
- **Take-Profit**: "If the price rises 10% from my buy price, sell automatically to lock in gains"

These are essential for risk management.

In [ ]:
class SMAWithStopLoss(bt.Strategy):
    """
    SMA Crossover with stop-loss and take-profit.
    """
    params = (
        ('fast_period', 50),
        ('slow_period', 200),
        ('stop_loss', 0.05),     # 5% stop-loss
        ('take_profit', 0.15),   # 15% take-profit
    )
    
    def __init__(self):
        self.sma_fast = bt.indicators.SMA(self.data.close, period=self.params.fast_period)
        self.sma_slow = bt.indicators.SMA(self.data.close, period=self.params.slow_period)
        self.crossover = bt.indicators.CrossOver(self.sma_fast, self.sma_slow)
        self.buy_price = None
    
    def next(self):
        if not self.position:
            # Entry: buy on golden cross
            if self.crossover > 0:
                self.buy()
                self.buy_price = self.data.close[0]
        else:
            # Exit conditions
            current_price = self.data.close[0]
            change = (current_price - self.buy_price) / self.buy_price
            
            # Stop-loss: price dropped too much
            if change < -self.params.stop_loss:
                self.sell()
                self.buy_price = None
            
            # Take-profit: price rose enough
            elif change > self.params.take_profit:
                self.sell()
                self.buy_price = None
            
            # Death cross: trend reversed
            elif self.crossover < 0:
                self.sell()
                self.buy_price = None

# Run
cerebro4 = bt.Cerebro()
cerebro4.adddata(bt.feeds.PandasData(dataname=aapl))
cerebro4.addstrategy(SMAWithStopLoss)
cerebro4.broker.setcash(100_000)
cerebro4.broker.setcommission(commission=0.001)
cerebro4.addsizer(bt.sizers.PercentSizer, percents=95)
cerebro4.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.04/252)
cerebro4.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

print(f'Start: ${cerebro4.broker.getvalue():,.2f}')
results4 = cerebro4.run()
final4 = cerebro4.broker.getvalue()
print(f'End:   ${final4:,.2f}')
print(f'Return: {(final4/100000-1)*100:.1f}%')

s4 = results4[0]
print(f'Sharpe: {s4.analyzers.sharpe.get_analysis().get("sharperatio", "N/A")}')
print(f'Max DD: {s4.analyzers.drawdown.get_analysis().max.drawdown:.1f}%')
print(f'\nStop-loss protects against big losses.')
print(f'Take-profit locks in gains before they disappear.')

### Backtrader Summary

**Pros:**
- Very flexible — can model almost any strategy
- Built-in indicators (100+ available)
- Multi-asset support
- Realistic order execution
- Built-in plotting

**Cons:**
- Verbose — lots of boilerplate code
- Slower than vectorized approaches
- Learning curve for the class-based architecture
- No longer actively maintained (but still widely used)

**Use when:** You need realistic simulation with complex order logic, stop-losses, portfolio rebalancing.

---
---
# Part 3: vectorbt
---
---

## Overview

**vectorbt** is a **vectorized** backtesting library. Instead of looping day by day, it processes everything at once using NumPy arrays.

### Key Differences from Backtrader

| | Backtrader | vectorbt |
|---|---|---|
| **Approach** | Loop through each day | Process all days at once |
| **Speed** | Slow (seconds to minutes) | Very fast (milliseconds) |
| **Code style** | Class-based (OOP) | Functional (chain operations) |
| **Parameter optimization** | Manual loops | Built-in, parallelized |
| **Best for** | Complex, realistic strategies | Quick testing, parameter sweeps |

### When to Use vectorbt
- You want to test **many parameter combinations** quickly
- You have a **simple signal-based strategy** (buy/sell based on conditions)
- You want **fast iteration** during strategy development

In [ ]:
# ===================================================================
# VECTORBT — Example 1: SMA Crossover
# ===================================================================

import vectorbt as vbt

# Get AAPL close prices
close = aapl['Close']
if isinstance(close, pd.DataFrame):
    close = close.squeeze()  # Convert single-column DataFrame to Series

# Calculate moving averages
fast_ma = vbt.MA.run(close, window=50)
slow_ma = vbt.MA.run(close, window=200)

# Generate entry/exit signals
# Entry: fast MA crosses above slow MA
# Exit: fast MA crosses below slow MA
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

# Run backtest — ONE LINE!
portfolio = vbt.Portfolio.from_signals(
    close,
    entries,
    exits,
    init_cash=100_000,
    fees=0.001,  # 0.1% commission
    freq='1D'
)

# Print performance
print('VECTORBT — SMA 50/200 Crossover on AAPL')
print('=' * 50)
print(f'Total Return:     {portfolio.total_return() * 100:.1f}%')
print(f'Annual Return:    {portfolio.annualized_return() * 100:.1f}%')
print(f'Sharpe Ratio:     {portfolio.sharpe_ratio():.3f}')
print(f'Max Drawdown:     {portfolio.max_drawdown() * 100:.1f}%')
print(f'Total Trades:     {portfolio.total_trades()}')
print(f'Win Rate:         {portfolio.win_rate() * 100:.1f}%')

In [ ]:
# vectorbt has beautiful built-in plots

# Plot portfolio value over time
portfolio.plot().show()

print('The plot shows portfolio value with buy/sell markers.')

In [ ]:
# Print full stats report
print(portfolio.stats())

## vectorbt — Example 2: Parameter Optimization

**This is vectorbt's killer feature.** You can test THOUSANDS of parameter combinations at once.

Question: "What's the best combination of fast and slow SMA periods?"

Instead of testing one by one, vectorbt tests them ALL simultaneously.

In [ ]:
# ===================================================================
# VECTORBT — Parameter Optimization
# ===================================================================
# Test ALL combinations of:
#   fast_period: 10, 20, 30, 40, 50, 60
#   slow_period: 100, 120, 150, 180, 200, 250
# That's 6 × 6 = 36 backtests at once!

fast_windows = [10, 20, 30, 40, 50, 60]
slow_windows = [100, 120, 150, 180, 200, 250]

# Calculate all MAs at once
fast_mas = vbt.MA.run(close, window=fast_windows)
slow_mas = vbt.MA.run(close, window=slow_windows)

# Generate all entry/exit combinations
# This creates a 2D grid of signals for every (fast, slow) combination
entries = fast_mas.ma_crossed_above(slow_mas)
exits = fast_mas.ma_crossed_below(slow_mas)

# Run ALL 36 backtests simultaneously
pf = vbt.Portfolio.from_signals(
    close, entries, exits,
    init_cash=100_000,
    fees=0.001,
    freq='1D'
)

# Get returns for all combinations
returns_grid = pf.total_return()

print('Total Returns for all (fast_period, slow_period) combinations:')
print('=' * 60)

# Reshape into a readable table
if isinstance(returns_grid, pd.Series):
    returns_df = returns_grid.unstack() * 100
    print(returns_df.round(1).to_string())
    
    # Find the best combination
    best_idx = returns_grid.idxmax()
    print(f'\nBest combination: fast={best_idx[0]}, slow={best_idx[1]}')
    print(f'Best return: {returns_grid.max()*100:.1f}%')
else:
    print(f'Return: {returns_grid*100:.1f}%')

In [ ]:
# Heatmap of returns by parameter combination

if isinstance(returns_grid, pd.Series):
    import seaborn as sns
    
    fig, ax = plt.subplots(figsize=(10, 8))
    returns_heatmap = returns_grid.unstack() * 100
    sns.heatmap(returns_heatmap, annot=True, fmt='.1f', cmap='RdYlGn',
                center=0, ax=ax)
    ax.set_title('Total Return (%) by SMA Parameters', fontsize=14)
    ax.set_xlabel('Slow MA Period')
    ax.set_ylabel('Fast MA Period')
    plt.tight_layout()
    plt.show()
    
    print('Green = profitable, Red = losing')
    print('This helps you find the best parameters WITHOUT overfitting.')

## vectorbt — Example 3: RSI Strategy

vectorbt has built-in RSI support too.

In [ ]:
# RSI Strategy in vectorbt — just a few lines

rsi = vbt.RSI.run(close, window=14)

# Buy when RSI crosses below 30 (oversold)
# Sell when RSI crosses above 70 (overbought)
entries_rsi = rsi.rsi_crossed_below(30)
exits_rsi = rsi.rsi_crossed_above(70)

pf_rsi = vbt.Portfolio.from_signals(
    close, entries_rsi, exits_rsi,
    init_cash=100_000, fees=0.001, freq='1D'
)

print('VECTORBT — RSI Mean Reversion on AAPL')
print('=' * 50)
print(f'Total Return:     {pf_rsi.total_return() * 100:.1f}%')
print(f'Sharpe Ratio:     {pf_rsi.sharpe_ratio():.3f}')
print(f'Max Drawdown:     {pf_rsi.max_drawdown() * 100:.1f}%')
print(f'Total Trades:     {pf_rsi.total_trades()}')
print(f'Win Rate:         {pf_rsi.win_rate() * 100:.1f}%')

In [ ]:
# RSI parameter optimization — test different thresholds
# What's the best buy/sell RSI level?

entry_thresholds = [20, 25, 30, 35, 40]
exit_thresholds = [60, 65, 70, 75, 80]

rsi_val = vbt.RSI.run(close, window=14).rsi

results_rsi = []
for entry_th in entry_thresholds:
    for exit_th in exit_thresholds:
        ent = rsi_val.vbt.crossed_below(entry_th)
        ext = rsi_val.vbt.crossed_above(exit_th)
        
        pf_temp = vbt.Portfolio.from_signals(
            close, ent, ext,
            init_cash=100_000, fees=0.001, freq='1D'
        )
        results_rsi.append({
            'Buy Below': entry_th,
            'Sell Above': exit_th,
            'Return (%)': round(pf_temp.total_return() * 100, 1),
            'Sharpe': round(pf_temp.sharpe_ratio(), 3),
            'Trades': pf_temp.total_trades()
        })

rsi_results_df = pd.DataFrame(results_rsi)
print('RSI Parameter Optimization Results:')
print(rsi_results_df.sort_values('Sharpe', ascending=False).head(10).to_string(index=False))

## vectorbt — Example 4: Multi-Stock Portfolio

vectorbt can also handle multiple stocks simultaneously.

In [ ]:
# Download multiple stocks
multi_close = yf.download(TICKERS, start='2020-01-01', end='2025-12-31', auto_adjust=False)['Adj Close']

# Apply SMA crossover to ALL stocks at once
fast = vbt.MA.run(multi_close, window=50)
slow = vbt.MA.run(multi_close, window=200)

entries_multi = fast.ma_crossed_above(slow)
exits_multi = fast.ma_crossed_below(slow)

# Backtest each stock independently
pf_multi = vbt.Portfolio.from_signals(
    multi_close, entries_multi, exits_multi,
    init_cash=100_000, fees=0.001, freq='1D'
)

# Compare all stocks
print('SMA 50/200 Crossover — Per Stock Performance')
print('=' * 60)
for ticker in TICKERS:
    try:
        ret = pf_multi.total_return()[ticker] * 100
        sr = pf_multi.sharpe_ratio()[ticker]
        dd = pf_multi.max_drawdown()[ticker] * 100
        print(f'  {ticker}: Return={ret:.1f}%, Sharpe={sr:.3f}, MaxDD={dd:.1f}%')
    except Exception:
        pass

### vectorbt Summary

**Pros:**
- Extremely fast (10-100x faster than Backtrader)
- Parameter optimization is trivial
- Beautiful built-in visualizations (Plotly)
- Concise code (backtest in 5 lines)
- Multi-asset support

**Cons:**
- Harder to model complex order logic (stop-loss, bracket orders)
- Less intuitive for beginners (vectorized thinking)
- Documentation can be sparse

**Use when:** You want fast iteration, parameter sweeps, or simple signal-based strategies.

---
---
# Part 4: backtesting.py
---
---

## Overview

**backtesting.py** is a lightweight, easy-to-use backtesting library. It's the simplest library to learn and produces beautiful interactive reports.

### Key Features
- Very simple API — easy to learn
- Interactive HTML reports with Bokeh plots
- Built-in optimization
- Supports stop-loss and take-profit
- Lightweight (no heavy dependencies)

In [ ]:
# ===================================================================
# BACKTESTING.PY — Example 1: SMA Crossover
# ===================================================================

from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import SMA

# backtesting.py requires column names: Open, High, Low, Close, Volume
# Our data already has these!

# Prepare data — backtesting.py needs a specific format
bt_df = aapl[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
if isinstance(bt_df.columns, pd.MultiIndex):
    bt_df.columns = bt_df.columns.get_level_values(0)

class SMACross(Strategy):
    # Parameters (can be optimized later)
    n1 = 50   # Fast SMA period
    n2 = 200  # Slow SMA period
    
    def init(self):
        # Calculate indicators using self.I()
        # self.I() registers the indicator for plotting
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        # Buy when fast crosses above slow
        if crossover(self.sma1, self.sma2):
            self.buy()
        # Sell when fast crosses below slow
        elif crossover(self.sma2, self.sma1):
            self.position.close()

# Run backtest
bt_result = Backtest(bt_df, SMACross, cash=100_000, commission=0.001)
stats = bt_result.run()

print('BACKTESTING.PY — SMA Crossover on AAPL')
print('=' * 50)
print(stats[['Start', 'End', 'Duration',
             'Return [%]', 'Buy & Hold Return [%]',
             'Max. Drawdown [%]', 'Sharpe Ratio',
             '# Trades', 'Win Rate [%]']].to_string())

In [ ]:
# Interactive plot — this generates a beautiful HTML report
# It shows: price, indicators, buy/sell points, equity curve, drawdown

bt_result.plot()
print('An interactive plot should appear above (or open in browser).')
print('You can zoom, pan, and hover over data points!')

## backtesting.py — Example 2: Built-in Optimization

backtesting.py has built-in parameter optimization — just call `.optimize()`.

In [ ]:
# Optimize SMA parameters
# Test all combinations of n1 (10-60) and n2 (100-250)

optimized_stats = bt_result.optimize(
    n1=range(10, 70, 10),      # Fast: 10, 20, 30, 40, 50, 60
    n2=range(100, 260, 30),     # Slow: 100, 130, 160, 190, 220, 250
    maximize='Sharpe Ratio',    # Optimize for best Sharpe Ratio
    constraint=lambda p: p.n1 < p.n2  # Fast must be shorter than slow
)

print('OPTIMIZED PARAMETERS')
print('=' * 50)
print(f'Best fast SMA:  {optimized_stats._strategy.n1}')
print(f'Best slow SMA:  {optimized_stats._strategy.n2}')
print(f'Sharpe Ratio:   {optimized_stats["Sharpe Ratio"]:.3f}')
print(f'Return:         {optimized_stats["Return [%]"]:.1f}%')
print(f'Max Drawdown:   {optimized_stats["Max. Drawdown [%]"]:.1f}%')

## backtesting.py — Example 3: Strategy with Stop-Loss / Take-Profit

backtesting.py makes stop-loss and take-profit very easy.

In [ ]:
class SMAWithRisk(Strategy):
    n1 = 50
    n2 = 200
    sl = 0.05   # 5% stop-loss
    tp = 0.15   # 15% take-profit
    
    def init(self):
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        if crossover(self.sma1, self.sma2):
            if not self.position:
                price = self.data.Close[-1]
                self.buy(
                    sl=price * (1 - self.sl),   # Stop-loss price
                    tp=price * (1 + self.tp),   # Take-profit price
                )
        elif crossover(self.sma2, self.sma1):
            self.position.close()

bt_risk = Backtest(bt_df, SMAWithRisk, cash=100_000, commission=0.001)
stats_risk = bt_risk.run()

print('SMA Crossover WITH Stop-Loss (5%) and Take-Profit (15%)')
print('=' * 50)
print(stats_risk[['Return [%]', 'Max. Drawdown [%]', 'Sharpe Ratio',
                   '# Trades', 'Win Rate [%]']].to_string())

### backtesting.py Summary

**Pros:**
- Simplest API of all libraries
- Beautiful interactive HTML plots
- Easy built-in optimization
- Stop-loss/take-profit support
- Actively maintained

**Cons:**
- Single asset only (no multi-stock portfolios)
- Less flexible than Backtrader for complex strategies
- No built-in portfolio management

**Use when:** Quick prototyping of single-stock strategies. Great for learning and experimentation.

---
---
# Part 5: Head-to-Head Comparison
---
---

Let's run the **same strategy** (SMA 50/200 Crossover on AAPL) in all libraries and compare.

In [ ]:
# ===================================================================
# HEAD-TO-HEAD COMPARISON
# Same strategy, same data, different libraries
# ===================================================================

import time

results_comparison = []

# --- Manual (pandas) ---
t0 = time.time()
df_m = aapl[['Close']].copy()
df_m.columns = ['close']
df_m['sma_f'] = df_m['close'].rolling(50).mean()
df_m['sma_s'] = df_m['close'].rolling(200).mean()
df_m['pos'] = (df_m['sma_f'] > df_m['sma_s']).astype(int).shift(1)
df_m['ret'] = df_m['close'].pct_change() * df_m['pos']
df_m = df_m.dropna()
manual_total = (1 + df_m['ret']).prod() - 1
t_manual = time.time() - t0
results_comparison.append({'Library': 'Manual (pandas)', 'Return (%)': round(manual_total*100, 1), 'Time (s)': round(t_manual, 3)})

# --- vectorbt ---
t0 = time.time()
c = aapl['Close']
if isinstance(c, pd.DataFrame): c = c.squeeze()
f_ma = vbt.MA.run(c, window=50)
s_ma = vbt.MA.run(c, window=200)
pf_v = vbt.Portfolio.from_signals(c, f_ma.ma_crossed_above(s_ma), f_ma.ma_crossed_below(s_ma),
                                   init_cash=100_000, fees=0.001, freq='1D')
vbt_total = pf_v.total_return()
t_vbt = time.time() - t0
results_comparison.append({'Library': 'vectorbt', 'Return (%)': round(vbt_total*100, 1), 'Time (s)': round(t_vbt, 3)})

# --- Backtrader ---
t0 = time.time()
cb = bt.Cerebro()
cb.adddata(bt.feeds.PandasData(dataname=aapl))
cb.addstrategy(SMACrossover)
cb.broker.setcash(100_000)
cb.broker.setcommission(commission=0.001)
cb.addsizer(bt.sizers.PercentSizer, percents=95)
cb.run()
bt_total = (cb.broker.getvalue() / 100_000 - 1)
t_bt = time.time() - t0
results_comparison.append({'Library': 'Backtrader', 'Return (%)': round(bt_total*100, 1), 'Time (s)': round(t_bt, 3)})

# --- backtesting.py ---
t0 = time.time()
btpy = Backtest(bt_df, SMACross, cash=100_000, commission=0.001)
stats_btpy = btpy.run()
btpy_total = stats_btpy['Return [%]']
t_btpy = time.time() - t0
results_comparison.append({'Library': 'backtesting.py', 'Return (%)': round(btpy_total, 1), 'Time (s)': round(t_btpy, 3)})

# Display comparison
comp_df = pd.DataFrame(results_comparison)
print('HEAD-TO-HEAD: SMA 50/200 Crossover on AAPL')
print('=' * 50)
print(comp_df.to_string(index=False))
print(f'\nNote: Returns differ slightly due to position sizing and execution differences.')

---
---
# Part 6: Which Library Should You Use?
---
---

## Decision Guide

```
What are you trying to do?
│
├── Quick prototype / learning?
│   └── Use backtesting.py (simplest, pretty plots)
│
├── Test many parameters fast?
│   └── Use vectorbt (fastest, built-in optimization)
│
├── Complex multi-asset portfolio with realistic execution?
│   └── Use Backtrader (most flexible, event-driven)
│
└── Need RL/AI integration?
    └── Use FinRL (built on top of these, RL-ready)
```

## For Your Group Project

**Recommended approach:**

1. **Use vectorbt** for initial strategy development and parameter optimization
   - Fast iteration, easy to try many ideas
   - Parameter heatmaps help you understand what works

2. **Use Backtrader** for the final, realistic backtest
   - Multi-asset portfolio support
   - Realistic execution with commissions
   - Periodic rebalancing

3. **Use FinRL** if going the RL path
   - Built-in stock trading environment
   - Handles RL agent training and testing

## Feature Comparison Table

| Feature | Manual | Backtrader | vectorbt | backtesting.py |
|---------|--------|------------|----------|----------------|
| Ease of use | Medium | Medium | Medium | Easy |
| Speed | Fast | Slow | Very Fast | Fast |
| Multi-asset | Manual | Yes | Yes | No |
| Stop-loss/TP | Manual | Yes | Limited | Yes |
| Parameter optimization | Manual | Manual | Built-in | Built-in |
| Realistic execution | No | Yes | Limited | Yes |
| Interactive plots | No | Basic | Plotly | Bokeh |
| Portfolio rebalancing | Manual | Yes | Manual | No |
| Built-in indicators | No | 100+ | Yes | Basic |
| Actively maintained | N/A | No | Yes | Yes |

---
---
# Part 7: Common Backtesting Mistakes
---
---

These mistakes will make your backtest results WRONG. Avoid them!

### 1. Look-Ahead Bias
**Problem:** Using future data to make decisions.

```python
# WRONG — you can't know today's close price to decide to buy today
if today_close > sma:
    buy_today()

# RIGHT — use yesterday's signal to trade today
if yesterday_close > yesterday_sma:
    buy_today()
```

### 2. No Train/Test Split
**Problem:** Optimizing and testing on the same data.

```python
# WRONG
optimize_on(all_data)  # Find best parameters
test_on(all_data)      # Test on same data → falsely great results

# RIGHT
optimize_on(data_2020_2023)  # Find parameters
test_on(data_2024_2025)      # Test on UNSEEN data
```

### 3. Ignoring Transaction Costs
**Problem:** A strategy that trades 100 times/day looks great without fees, terrible with them.

```python
# Always include commission
cerebro.broker.setcommission(commission=0.001)  # 0.1% per trade
```

### 4. Survivorship Bias
**Problem:** Only testing on stocks that exist today. Companies that went bankrupt are excluded, making results look better.

### 5. Overfitting
**Problem:** Tuning parameters until they perfectly fit historical data, but they fail on new data.

**Rule of thumb:** If your strategy has more than 5 parameters, you're probably overfitting.

### 6. Not Accounting for Slippage
**Problem:** Assuming you can always buy/sell at the exact price you want. In reality, the price moves between your decision and execution.

```python
# Backtrader slippage
cerebro.broker.set_slippage_perc(perc=0.001)  # 0.1% slippage
```

In [ ]:
# ===================================================================
# DEMONSTRATION: Impact of transaction costs
# ===================================================================
# Same strategy, different fee levels

close_data = aapl['Close']
if isinstance(close_data, pd.DataFrame):
    close_data = close_data.squeeze()

f = vbt.MA.run(close_data, window=50)
s = vbt.MA.run(close_data, window=200)
ent = f.ma_crossed_above(s)
ext = f.ma_crossed_below(s)

fee_levels = [0, 0.001, 0.005, 0.01, 0.02]
fee_results = []

for fee in fee_levels:
    pf_fee = vbt.Portfolio.from_signals(close_data, ent, ext,
                                        init_cash=100_000, fees=fee, freq='1D')
    fee_results.append({
        'Fee (%)': f'{fee*100:.1f}%',
        'Total Return (%)': round(pf_fee.total_return()*100, 1),
        'Sharpe': round(pf_fee.sharpe_ratio(), 3)
    })

print('Impact of Transaction Costs on Returns')
print('=' * 50)
print(pd.DataFrame(fee_results).to_string(index=False))
print(f'\nHigher fees eat into returns — especially for frequent trading strategies!')

In [ ]:
print('NOTEBOOK COMPLETE!')
print('=' * 50)
print('Libraries covered:')
print('  1. Manual (pandas) — understand the fundamentals')
print('  2. Backtrader — complex, realistic, multi-asset')
print('  3. vectorbt — fast, optimized, parameter sweeps')
print('  4. backtesting.py — simple, pretty, quick prototyping')
print()
print('For your group project, recommended workflow:')
print('  → Develop ideas with vectorbt (fast iteration)')
print('  → Final backtest with Backtrader (realistic, multi-asset)')
print('  → Use backtesting.py for pretty report charts')